# Bidirectional LSTM from Scratch

**Goal:** understand sequence transduction and temporal memory tracking by manually
implementing the four LSTM gate equations (input, forget, candidate, output) and
combining a forward and a backward recurrence into a single bidirectional layer.

We work with fixed-length sequences and only use raw PyTorch tensor ops
(`torch.sigmoid`, `torch.tanh`, matmuls) for the core recurrence — no `nn.RNN` /
`nn.LSTM` calls inside our implementation. At the end we validate our output
against `nn.LSTM(bidirectional=True)` with matching weights, on both shape and
value.

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(0)
print(torch.__version__)

2.13.0+cu130


## 1. The LSTM gate equations

At each timestep $t$, given input $x_t$, previous hidden state $h_{t-1}$ and
previous cell state $c_{t-1}$, a single-direction LSTM computes four gates:

$$
\begin{aligned}
i_t &= \sigma(W_{ii} x_t + b_{ii} + W_{hi} h_{t-1} + b_{hi}) && \text{input gate}\\
f_t &= \sigma(W_{if} x_t + b_{if} + W_{hf} h_{t-1} + b_{hf}) && \text{forget gate}\\
g_t &= \tanh(W_{ig} x_t + b_{ig} + W_{hg} h_{t-1} + b_{hg}) && \text{candidate cell state}\\
o_t &= \sigma(W_{io} x_t + b_{io} + W_{ho} h_{t-1} + b_{ho}) && \text{output gate}
\end{aligned}
$$

and then updates the cell state and hidden state:

$$
c_t = f_t \odot c_{t-1} + i_t \odot g_t \qquad\qquad h_t = o_t \odot \tanh(c_t)
$$

Intuitively:
- **Forget gate** $f_t$ decides how much of the old memory $c_{t-1}$ to keep.
- **Input gate** $i_t$ decides how much of the new candidate $g_t$ to write in.
- **Candidate** $g_t$ is the "proposed" new content for the cell state.
- **Output gate** $o_t$ decides how much of the (updated) cell state to expose
  as the hidden state $h_t$.

PyTorch's `nn.LSTM` stacks the four gates' weights into single matrices
`weight_ih` and `weight_hh` of shape `(4*hidden_size, input_size)` and
`(4*hidden_size, hidden_size)`, in the order **i, f, g, o**. We'll match that
layout so we can borrow `nn.LSTM`'s weights directly and compare outputs.

## 2. A single-direction recurrence (input → forget → candidate → output)

This is the core loop: step through time, compute the four gates from the
stacked weight matrices, update $c_t$ and $h_t$, and stash $h_t$. Passing
`reverse=True` walks the sequence back-to-front (used for the backward
direction later) but always *writes* each output to its true timestep index,
so the returned tensor stays in original time order.

In [2]:
def lstm_direction(x, W_ih, W_hh, b_ih, b_hh, hidden_size, reverse=False):
    """
    Run one direction (forward or backward) of an LSTM over a fixed-length
    sequence using only raw tensor ops.

    x:      (batch, seq_len, input_size)
    W_ih:   (4*hidden_size, input_size)   -- stacked [W_ii; W_if; W_ig; W_io]
    W_hh:   (4*hidden_size, hidden_size)  -- stacked [W_hi; W_hf; W_hg; W_ho]
    b_ih, b_hh: (4*hidden_size,)
    reverse: if True, recurrence runs from t=T-1 down to t=0 (still returns
             outputs indexed by original timestep, for easy concatenation)

    Returns: (batch, seq_len, hidden_size) hidden states at every timestep
    """
    batch_size, seq_len, _ = x.shape
    device = x.device

    h_t = torch.zeros(batch_size, hidden_size, device=device)
    c_t = torch.zeros(batch_size, hidden_size, device=device)

    time_steps = range(seq_len - 1, -1, -1) if reverse else range(seq_len)
    outputs = [None] * seq_len

    for t in time_steps:
        x_t = x[:, t, :]                                   # (batch, input_size)

        # one matmul each for the four stacked gates, PyTorch gate order: i, f, g, o
        gates = x_t @ W_ih.T + b_ih + h_t @ W_hh.T + b_hh   # (batch, 4*hidden_size)
        i_gate, f_gate, g_gate, o_gate = gates.chunk(4, dim=1)

        i_t = torch.sigmoid(i_gate)   # input gate
        f_t = torch.sigmoid(f_gate)   # forget gate
        g_t = torch.tanh(g_gate)      # candidate cell state
        o_t = torch.sigmoid(o_gate)   # output gate

        c_t = f_t * c_t + i_t * g_t   # new cell state
        h_t = o_t * torch.tanh(c_t)   # new hidden state

        outputs[t] = h_t

    return torch.stack(outputs, dim=1)   # (batch, seq_len, hidden_size)

## 3. Bidirectional layer

Run the recurrence once forward over the sequence and once over the reversed
sequence, then **concatenate** the forward and backward hidden states at each
timestep along the feature dimension. This is exactly what
`nn.LSTM(bidirectional=True)` does, giving an output of shape
`(batch, seq_len, 2*hidden_size)`.

In [3]:
class ManualBiLSTM:
    """Bidirectional single-layer LSTM built from raw tensor ops."""

    def __init__(self, W_ih_f, W_hh_f, b_ih_f, b_hh_f,
                       W_ih_b, W_hh_b, b_ih_b, b_hh_b, hidden_size):
        self.fwd_params = (W_ih_f, W_hh_f, b_ih_f, b_hh_f)
        self.bwd_params = (W_ih_b, W_hh_b, b_ih_b, b_hh_b)
        self.hidden_size = hidden_size

    def __call__(self, x):
        fwd_out = lstm_direction(x, *self.fwd_params, self.hidden_size, reverse=False)
        bwd_out = lstm_direction(x, *self.bwd_params, self.hidden_size, reverse=True)
        # concatenate forward and backward hidden states at each timestep
        return torch.cat([fwd_out, bwd_out], dim=2)   # (batch, seq_len, 2*hidden_size)

## 4. Validation against `nn.LSTM(bidirectional=True)`

We build a reference `nn.LSTM`, pull its (randomly-initialized) weights out,
and feed them into our manual implementation on the exact same fixed-length
input batch. If our gate math is right, shapes and values should match to
floating-point precision.

In [4]:
batch_size   = 4
seq_len      = 6     # fixed-length sequences
input_size   = 5
hidden_size  = 8

x = torch.randn(batch_size, seq_len, input_size)

ref = nn.LSTM(input_size, hidden_size, batch_first=True, bidirectional=True)

manual_bilstm = ManualBiLSTM(
    ref.weight_ih_l0,          ref.weight_hh_l0,          ref.bias_ih_l0,          ref.bias_hh_l0,
    ref.weight_ih_l0_reverse,  ref.weight_hh_l0_reverse,  ref.bias_ih_l0_reverse,  ref.bias_hh_l0_reverse,
    hidden_size,
)

with torch.no_grad():
    manual_out = manual_bilstm(x)
    ref_out, (h_n, c_n) = ref(x)

print("manual output shape:  ", manual_out.shape)
print("nn.LSTM output shape: ", ref_out.shape)

manual output shape:   torch.Size([4, 6, 16])
nn.LSTM output shape:  torch.Size([4, 6, 16])


In [5]:
max_abs_diff = (manual_out - ref_out).abs().max().item()
values_match = torch.allclose(manual_out, ref_out, atol=1e-6)
shapes_match = manual_out.shape == ref_out.shape

print(f"max abs diff:  {max_abs_diff:.3e}")
print(f"shapes match:  {shapes_match}")
print(f"values match:  {values_match}")

assert shapes_match, "Output shapes don't match nn.LSTM!"
assert values_match, "Output values don't match nn.LSTM!"
print("\nManual bidirectional LSTM matches nn.LSTM(bidirectional=True).")

max abs diff:  5.960e-08
shapes match:  True
values match:  True

Manual bidirectional LSTM matches nn.LSTM(bidirectional=True).


### Sanity check: final hidden states line up too

`nn.LSTM` also returns the final hidden state `h_n` with shape
`(num_directions, batch, hidden_size)`. For the forward direction that's just
our forward output at the last timestep; for the backward direction it's our
backward output at timestep 0 (since the backward pass finishes there).

In [6]:
fwd_h_n_manual = manual_out[:, -1, :hidden_size]     # forward dir, last timestep
bwd_h_n_manual = manual_out[:, 0, hidden_size:]      # backward dir, first timestep (its own t=0)

fwd_h_n_ref = h_n[0]   # (batch, hidden_size)
bwd_h_n_ref = h_n[1]   # (batch, hidden_size)

print("forward h_n matches: ", torch.allclose(fwd_h_n_manual, fwd_h_n_ref, atol=1e-6))
print("backward h_n matches:", torch.allclose(bwd_h_n_manual, bwd_h_n_ref, atol=1e-6))

forward h_n matches:  True
backward h_n matches: True


## 5. Takeaways

- The four gates (**i**nput, **f**orget, **g** candidate, **o**utput) are each
  just an affine map of `[x_t, h_{t-1}]` through a nonlinearity — three of the
  four use sigmoid to act as "soft switches" between 0 and 1, and the
  candidate uses tanh to propose new content in `[-1, 1]`.
- The cell state $c_t$ is the layer's long-term memory: it's updated by an
  elementwise blend (`forget * old + input * candidate`), which is what lets
  gradients flow across many timesteps without vanishing as fast as in a
  vanilla RNN.
- "Bidirectional" is just running this same recurrence twice — once forward,
  once on the time-reversed sequence — and concatenating the two hidden
  states at each timestep. Each direction has its own independent set of
  weights, so the backward pass isn't just a mirror of the forward one, it's
  a separately-trained model of the reverse temporal dependencies.